In [3]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/Users/zurabi/Desktop/ML_FINAL_PROJECT/ML_FINAL_PROJECT")

In [4]:
import zipfile

zip_path = PROJECT_DIR / "walmart-recruiting-store-sales-forecasting.zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(PROJECT_DIR)

In [5]:
train = pd.read_csv(PROJECT_DIR / "train.csv.zip")
test = pd.read_csv(PROJECT_DIR / "test.csv.zip")
features = pd.read_csv(PROJECT_DIR / "features.csv.zip")
stores = pd.read_csv(PROJECT_DIR / "stores.csv")
sample_submission = pd.read_csv(PROJECT_DIR / "sampleSubmission.csv.zip")

In [6]:
print(f"train shape: {train.shape}")
print(train.dtypes)
train.head()

train shape: (421570, 5)
Store             int64
Dept              int64
Date             object
Weekly_Sales    float64
IsHoliday          bool
dtype: object


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


In [7]:
print(f"test shape: {test.shape}")
test.head()

test shape: (115064, 4)


,Store,Dept,Date,IsHoliday
0,1,1,2012-11-02,False
1,1,1,2012-11-09,False
2,1,1,2012-11-16,False
3,1,1,2012-11-23,True
4,1,1,2012-11-30,False


In [8]:
print(f"features shape: {features.shape}")
features.head()

features shape: (8190, 12)


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [9]:
print(f"stores shape: {stores.shape}")
stores.head()

stores shape: (45, 3)


,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [10]:
train["Date"] = pd.to_datetime(train["Date"])
test["Date"] = pd.to_datetime(test["Date"])
features["Date"] = pd.to_datetime(features["Date"])

##### Here we compare train and test dates. test dates come after train dates

In [11]:
print("Train dates:", train["Date"].min(), "to", train["Date"].max())
print("Test dates:", test["Date"].min(), "to", test["Date"].max())
print("Features dates:", features["Date"].min(), "to", features["Date"].max())

Train dates: 2010-02-05 00:00:00 to 2012-10-26 00:00:00
Test dates: 2012-11-02 00:00:00 to 2013-07-26 00:00:00
Features dates: 2010-02-05 00:00:00 to 2013-07-26 00:00:00


In [14]:
df = train.merge(features, on=["Store", "Date", "IsHoliday"], how="left")
df = df.merge(stores, on="Store", how="left")

print(df.shape) #same as train shape in terms of rows. good
df.head()

(421570, 16)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   Store         421570 non-null  int64         
 1   Dept          421570 non-null  int64         
 2   Date          421570 non-null  datetime64[ns]
 3   Weekly_Sales  421570 non-null  float64       
 4   IsHoliday     421570 non-null  bool          
 5   Temperature   421570 non-null  float64       
 6   Fuel_Price    421570 non-null  float64       
 7   MarkDown1     150681 non-null  float64       
 8   MarkDown2     111248 non-null  float64       
 9   MarkDown3     137091 non-null  float64       
 10  MarkDown4     134967 non-null  float64       
 11  MarkDown5     151432 non-null  float64       
 12  CPI           421570 non-null  float64       
 13  Unemployment  421570 non-null  float64       
 14  Type          421570 non-null  object        
 15  Size          421

In [17]:
df.isna().mean().sort_values(ascending=False)

MarkDown2       0.736110
MarkDown4       0.679847
MarkDown3       0.674808
MarkDown1       0.642572
MarkDown5       0.640790
Store           0.000000
Dept            0.000000
Date            0.000000
Weekly_Sales    0.000000
IsHoliday       0.000000
Temperature     0.000000
Fuel_Price      0.000000
CPI             0.000000
Unemployment    0.000000
Type            0.000000
Size            0.000000
dtype: float64